# LangChain: calls, chains and chat prompts
Open this notebook from its `Code` folder and select the course Python kernel.
Run the setup once, then follow the sections when indicated in the slides.

## Setup
Load the course key using a path relative to this notebook's folder.

In [ ]:
from pathlib import Path
import yaml

key_file = Path("../../../llm_provider_api_key.yaml")
api_key = yaml.safe_load(key_file.read_text())["ollama"]["api_key"]

## 1. First call
Compare client creation and a call using the same provider, model and request.

In [ ]:
from ollama import Client

client = Client(
    host="https://ollama.com",
    headers={"Authorization": "Bearer " + api_key},
)
request = "Welcome a museum visitor in one sentence."
direct_reply = client.chat(
    model="gpt-oss:120b",
    messages=[{"role": "user", "content": request}],
)
direct_reply

LangChain returns an `AIMessage`. Inspect the object, then its text in `content`.

In [ ]:
from langchain_ollama import ChatOllama

llm = ChatOllama(
    model="gpt-oss:120b",
    base_url="https://ollama.com",
    client_kwargs={"headers": {"Authorization": "Bearer " + api_key}},
)

reply = llm.invoke(request)
reply

In [ ]:
reply.content

### Try on the agent
Open [`langchain_naive_agent.py`](../../../../virtual_human_project/vh_cognitive_architecture_server/agents/Module_1/Chapter_03/langchain_naive_agent.py).
Select its import in `vh_cognitive_server.py`, save and restart the cognitive server:

```python
from agents.Module_1.Chapter_03.langchain_naive_agent import response
```

Send a request and listen to the avatar's reply.

## 2. Chains
`|` passes one component's output to the next.
`StrOutputParser` extracts the text from the model's message.

In [ ]:
from langchain_core.output_parsers import StrOutputParser

text_chain = llm | StrOutputParser()
text_chain.invoke("Welcome a museum visitor in one sentence.")

### Generate an answer, then check it
The first model answers the user. The second checks that answer before we show it.
Our rule: if it discusses football, replace it with a refusal. Otherwise, return it unchanged.

In [ ]:
request = "Explain football in one sentence."
draft = text_chain.invoke(request)
print(draft)

In [ ]:
guardrail_rules = (
    "Check the text below. If it discusses football, replace it with exactly: "
    "I am not authorised to discuss football. "
    "Otherwise return the text unchanged. Return only the final text."
)
review_input = guardrail_rules + "\nText to check:\n" + draft
print(review_input)

In [ ]:
checked_answer = text_chain.invoke(review_input)
print(checked_answer)

### Connect the two calls
The middle step adds the checking rules to the first answer.
Later we will use chat templates to keep rules and text in separate messages.

In [ ]:
guarded_chain = (
    text_chain
    | (lambda text: guardrail_rules + "\nText to check:\n" + text)
    | text_chain
)
# guarded_chain.invoke(request)

### Try it
Try a football request and a request about another topic.
Check that only the football answer is replaced. You can edit the rule and compare results.
This model-based check can still make mistakes.

## 3. Conversations and chat prompts
Examples in a prompt can show a character's way of speaking.
The model continues the conversation without changing its weights.

In [ ]:
conversation = """Visitor: Hello!
PippoBot: Ahoy, visitor! Welcome aboard our museum.
Visitor: Can you explain an exhibit?
PippoBot: Arrr, matey! Let us uncover its story together.
Visitor: Can you help me explore?
PippoBot:"""

text_chain.invoke("Continue with PippoBot's next reply.\n" + conversation)

### Message objects
Chat models are tuned on conversations with roles and message boundaries.
LangChain provides objects for these roles; the provider applies the model's chat format.

- `SystemMessage`: instructions.
- `HumanMessage`: user input.
- `AIMessage`: an assistant reply or an example supplied by us.

In [ ]:
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage

messages = [
    SystemMessage("You are PippoBot, a pirate museum guide."),
    HumanMessage("Hello!"),
    AIMessage("Ahoy, visitor!"),
    HumanMessage("Can you help me explore?"),
]
for message in messages:
    message.pretty_print()

### Message templates: static and dynamic parts
Fixed text stays the same. Names inside braces are variables filled at each call.
A message template creates a message when we supply those values.

In [ ]:
from langchain_core.prompts import (
    ChatPromptTemplate, SystemMessagePromptTemplate, HumanMessagePromptTemplate,
)

system_template = SystemMessagePromptTemplate.from_template(
    "You are {character}. Speak in a {style} style."
)
system_template.format(character="PippoBot", style="pirate").pretty_print()

### One dictionary fills the chat prompt
ChatPromptTemplate combines the message templates. The dictionary supplies every dynamic value.
Inspect the resulting messages before the model call.

In [ ]:
human_template = HumanMessagePromptTemplate.from_template("{user_input}")
chat_prompt = ChatPromptTemplate.from_messages([system_template, human_template])
values = {
    "character": "PippoBot",
    "style": "pirate",
    "user_input": "Can you help me explore?",
}
print(chat_prompt.invoke(values).to_string())

In [ ]:
chat_chain = chat_prompt | text_chain
chat_chain.invoke(values)

### The same guardrail with chat templates
Put the checking rules in system and the text to check in human.
The rules stay fixed; the dictionary fills the dynamic `{text}` field.

In [ ]:
review_prompt = ChatPromptTemplate.from_messages([
    SystemMessagePromptTemplate.from_template(
        "Check the text. If it discusses football, reply exactly: "
        "I am not authorised to discuss football. "
        "Otherwise return the text unchanged. Return only the final text."
    ),
    HumanMessagePromptTemplate.from_template("{text}"),
])
review_values = {"text": "The football match ended 2-0."}
print(review_prompt.invoke(review_values).to_string())
review_chain = review_prompt | text_chain
review_chain.invoke(review_values)

### Generate an answer and review it
The first chain answers a request. Pass that answer to the review chain using a dictionary.

In [ ]:
answer_prompt = ChatPromptTemplate.from_messages([
    HumanMessagePromptTemplate.from_template("Answer in one sentence: {request}"),
])
answer_chain = answer_prompt | text_chain
answer = answer_chain.invoke({"request": "Explain football."})
review_chain.invoke({"text": answer})

### Exercise: style and current time
Choose a style in `style_prompt`. The message templates use a dictionary to fill style, time and user input.
The code supplies the computer's current time. Ask the bot what time it is and check its answer and style.

In [ ]:
style_prompt = "Reply briefly."

In [ ]:
from datetime import datetime

timed_prompt = ChatPromptTemplate.from_messages([
    SystemMessagePromptTemplate.from_template("{style} The current time is {time}."),
    HumanMessagePromptTemplate.from_template("{user_input}"),
])
timed_chain = timed_prompt | text_chain
values = {
    "style": style_prompt,
    "time": datetime.now().strftime("%H:%M"),
    "user_input": "What time is it?",
}
print(timed_prompt.invoke(values).to_string())
timed_chain.invoke(values)

### Try on the agent
In [`langchain_naive_agent.py`](../../../../virtual_human_project/vh_cognitive_architecture_server/agents/Module_1/Chapter_03/langchain_naive_agent.py),
set `STYLE_PROMPT` to your chosen instruction. Save and restart the cognitive server.
Ask the avatar what time it is. The agent supplies the current time on each call.